## Resources

Riths, I don't want to hand you another notebook that's likely to break after 40 minutes of setup.

You've already seen:

*   DistilBERT notebook
*   DeBERTa notebook
*   version conflicts
*   torchvision nonsense
*   accelerate nonsense

So let's do this properly.

### First: Clean Colab

**Cell 1**

In [ ]:
!pip uninstall -y transformers datasets accelerate torchvision torchaudio torchtext
!pip install -q \
torch==2.3.1 \
transformers==4.41.2 \
datasets==2.19.1 \
accelerate==0.30.1 \
evaluate==0.4.2 \
scikit-learn \
sentencepiece

Found existing installation: transformers 4.46.3
Uninstalling transformers-4.46.3:
  Successfully uninstalled transformers-4.46.3
Found existing installation: datasets 3.1.0
Uninstalling datasets-3.1.0:
  Successfully uninstalled datasets-3.1.0
Found existing installation: accelerate 1.1.1
Uninstalling accelerate-1.1.1:
  Successfully uninstalled accelerate-1.1.1


Okay, STOP.

This is actually good news.

Your code is fine.

The notebook is fine.

The dataset is fine.

The problem is **version mismatch**.

The error:

```text
cannot import name 'clear_device_cache'
from accelerate.utils.memory
```

means:

```text
transformers 4.41.x
expects
accelerate >= newer version

but

accelerate installed is older / corrupted
```

---

Run this SINGLE CELL:

In [ ]:
!pip uninstall -y transformers accelerate datasets

!pip install -q \
transformers==4.46.3 \
accelerate==1.1.1 \
datasets==3.1.0 \
evaluate==0.4.3 \
sentencepiece \
scikit-learn

Found existing installation: transformers 4.41.2
Uninstalling transformers-4.41.2:
  Successfully uninstalled transformers-4.41.2
Found existing installation: accelerate 0.30.1
Uninstalling accelerate-0.30.1:
  Successfully uninstalled accelerate-0.30.1
Found existing installation: datasets 2.19.1
Uninstalling datasets-2.19.1:
  Successfully uninstalled datasets-2.19.1


Then:

```python
import os
os.kill(os.getpid(), 9)
```

This FORCE restarts Colab.

---

In [ ]:
import os
os.kill(os.getpid(), 9)

After reconnecting:

Run ONLY:

In [ ]:
import transformers
import accelerate
import datasets
import torch

print("Transformers:", transformers.__version__)
print("Accelerate:", accelerate.__version__)
print("Datasets:", datasets.__version__)
print("Torch:", torch.__version__)

Transformers: 4.46.3
Accelerate: 1.1.1
Datasets: 3.1.0
Torch: 2.3.1+cu121


I expect something close to:

```text
Transformers: 4.46.3
Accelerate: 1.1.1
Datasets: 3.1.0
```

---

Then rerun notebook from Cell 3 onwards.

---

### Before continuing

Run:

In [ ]:
import transformers
import accelerate

print(transformers.__version__)
print(accelerate.__version__)

4.46.3
1.1.1


### Cell 3

In [ ]:
import torch
import transformers
import datasets
import accelerate

print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("Accelerate:", accelerate.__version__)

print("\nGPU:", torch.cuda.get_device_name(0))

Torch: 2.3.1+cu121
Transformers: 4.46.3
Datasets: 3.1.0
Accelerate: 1.1.1

GPU: Tesla T4


Expected:

```text
GPU: Tesla T4
```

### Cell 4

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Cell 5

In [ ]:
import pandas as pd

PATH = "/content/drive/MyDrive/jigsaw_data/train.csv"

df = pd.read_csv(PATH)

print(df.shape)

df.head()

(159571, 8)


,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0


### Cell 6

In [ ]:
LABELS = [
    "toxic",
    "severe_toxic",
    "obscene",
    "threat",
    "insult",
    "identity_hate"
]

df["labels"] = (
    df[LABELS]
    .astype(float)
    .values
    .tolist()
)

df = df[
    ["comment_text","labels"]
]

df.head()

,comment_text,labels
0,Explanation\nWhy the edits made under my usern...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0]"
1,D'aww! He matches this background colour I'm s...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0]"
2,"Hey man, I'm really not trying to edit war. It...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0]"
3,"""\nMore\nI can't make any real suggestions on ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0]"
4,"You, sir, are my hero. Any chance you remember...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0]"


In [ ]:
display(df.head())

### Cell 7

In [ ]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    df,
    test_size=0.1,
    random_state=42
)

print(train_df.shape)
print(val_df.shape)

(143613, 2)
(15958, 2)


### Cell 8

In [ ]:
from datasets import Dataset

train_ds = Dataset.from_pandas(
    train_df
)

val_ds = Dataset.from_pandas(
    val_df
)

print(train_ds)

Dataset({
    features: ['comment_text', 'labels', '__index_level_0__'],
    num_rows: 143613
})


### Cell 9

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "roberta-base"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


### Cell 10

In [ ]:
MAX_LEN = 256

def tokenize(batch):

    return tokenizer(
        batch["comment_text"],
        truncation=True,
        max_length=MAX_LEN
    )

train_ds = train_ds.map(
    tokenize,
    batched=True
)

val_ds = val_ds.map(
    tokenize,
    batched=True
)

Map:   0%|          | 0/143613 [00:00<?, ? examples/s]

Map:   0%|          | 0/15958 [00:00<?, ? examples/s]

### Cell 11

In [ ]:
train_ds = train_ds.remove_columns(
    ["comment_text"]
)

val_ds = val_ds.remove_columns(
    ["comment_text"]
)

### Cell 12

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=6,
    problem_type="multi_label_classification"
)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


### Cell 13

In [ ]:
import numpy as np
import torch

from sklearn.metrics import (
    f1_score
)

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    probs = torch.sigmoid(
        torch.tensor(logits)
    ).numpy()

    preds = (
        probs > 0.5
    ).astype(int)

    micro_f1 = f1_score(
        labels,
        preds,
        average="micro"
    )

    macro_f1 = f1_score(
        labels,
        preds,
        average="macro"
    )

    return {
        "micro_f1": micro_f1,
        "macro_f1": macro_f1
    }

### Cell 14

In [ ]:
from transformers import (
    TrainingArguments
)

training_args = TrainingArguments(

    output_dir="./safechat",

    learning_rate=2e-5,

    per_device_train_batch_size=8,

    per_device_eval_batch_size=8,

    num_train_epochs=3,

    weight_decay=0.01,

    fp16=True,

    evaluation_strategy="epoch",

    save_strategy="epoch",

    load_best_model_at_end=True,

    metric_for_best_model="micro_f1",

    greater_is_better=True,

    report_to="none"
)

/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


### Cell 15

In [ ]:
from transformers import (
    Trainer,
    DataCollatorWithPadding
)

data_collator = DataCollatorWithPadding(
    tokenizer
)

trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_ds,

    eval_dataset=val_ds,

    data_collator=data_collator,

    compute_metrics=compute_metrics
)

### Cell 16

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Micro F1,Macro F1
1,0.040100,0.039022,0.786838,0.569341
2,0.033300,0.039651,0.786713,0.651894
3,0.029000,0.041479,0.795002,0.669865


TrainOutput(global_step=53856, training_loss=0.03798924582173126, metrics={'train_runtime': 7432.331, 'train_samples_per_second': 57.968, 'train_steps_per_second': 7.246, 'total_flos': 4.4808847938506984e+16, 'train_loss': 0.03798924582173126, 'epoch': 3.0})

### Cell 17

In [ ]:
metrics = trainer.evaluate()
print(metrics)

{'eval_loss': 0.04147908464074135, 'eval_micro_f1': 0.7950021792822897, 'eval_macro_f1': 0.6698649291698838, 'eval_runtime': 64.4988, 'eval_samples_per_second': 247.416, 'eval_steps_per_second': 30.931, 'epoch': 3.0}


### Cell 18

In [ ]:
SAVE_PATH = "/content/drive/MyDrive/SafeChat_RoBERTa"

trainer.save_model(
    SAVE_PATH
)

tokenizer.save_pretrained(
    SAVE_PATH
)

('/content/drive/MyDrive/SafeChat_RoBERTa/tokenizer_config.json',
 '/content/drive/MyDrive/SafeChat_RoBERTa/special_tokens_map.json',
 '/content/drive/MyDrive/SafeChat_RoBERTa/vocab.json',
 '/content/drive/MyDrive/SafeChat_RoBERTa/merges.txt',
 '/content/drive/MyDrive/SafeChat_RoBERTa/added_tokens.json',
 '/content/drive/MyDrive/SafeChat_RoBERTa/tokenizer.json')

### Cell 19

In [ ]:
from transformers import pipeline

classifier = pipeline(

    "text-classification",

    model=SAVE_PATH,

    tokenizer=SAVE_PATH,

    top_k=None,

    device=0
)

### Cell 20

In [ ]:
LABELS = [
    "toxic",
    "severe_toxic",
    "obscene",
    "threat",
    "insult",
    "identity_hate"
]

def moderate(text):

    output = classifier(text)[0]

    print("\nTEXT:")
    print(text)

    unsafe = False

    for i, item in enumerate(output):

        score = item["score"]

        label = LABELS[i]

        print(
            f"{label:<15}: {score:.3f}"
        )

        if score > 0.5:
            unsafe = True

    print()

    if unsafe:
        print("🚨 UNSAFE")
    else:
        print("✅ SAFE")

### Cell 21

In [ ]:
moderate(
    "Hello my friend"
)

moderate(
    "You are an idiot"
)

moderate(
    "I will slit your throat"
)

moderate(
    "All muslims are terrorists"
)


TEXT:
Hello my friend
toxic          : 0.000
severe_toxic   : 0.000
obscene        : 0.000
threat         : 0.000
insult         : 0.000
identity_hate  : 0.000

✅ SAFE

TEXT:
You are an idiot
toxic          : 0.991
severe_toxic   : 0.978
obscene        : 0.713
threat         : 0.029
insult         : 0.007
identity_hate  : 0.002

🚨 UNSAFE

TEXT:
I will slit your throat
toxic          : 0.979
severe_toxic   : 0.957
obscene        : 0.190
threat         : 0.061
insult         : 0.056
identity_hate  : 0.033

🚨 UNSAFE

TEXT:
All muslims are terrorists
toxic          : 0.949
severe_toxic   : 0.908
obscene        : 0.247
threat         : 0.037
insult         : 0.027
identity_hate  : 0.010

🚨 UNSAFE


In [ ]:
from transformers import pipeline, AutoModelForSequenceClassification, AutoTokenizer
import torch

MODEL_PATH = "/content/drive/MyDrive/SafeChat_RoBERTa"

LABELS = [
    "toxic",
    "severe_toxic",
    "obscene",
    "threat",
    "insult",
    "identity_hate"
]

# Explicitly load model and tokenizer
model_loaded = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)
tokenizer_loaded = AutoTokenizer.from_pretrained(MODEL_PATH)

classifier = pipeline(
    "text-classification",
    model=model_loaded, # Pass the loaded model object
    tokenizer=tokenizer_loaded, # Pass the loaded tokenizer object
    top_k=None,
    device=0 if torch.cuda.is_available() else -1
)

def moderate(text):

    results = classifier(text)[0]

    print("\n" + "="*60)
    print("TEXT:")
    print(text)
    print("="*60)

    unsafe = False

    detected = []

    for i, item in enumerate(results):

        label = LABELS[i]
        score = item["score"]

        print(f"{label:<15}: {score:.4f}")

        if score > 0.50:
            unsafe = True
            detected.append(label)

    print("\nRESULT")

    if unsafe:
        print("🚨 UNSAFE")

        print("\nDetected Categories:")

        for d in detected:
            print("-", d)

        print("\nBLURRED MESSAGE")
        print("████████████████████████")

    else:
        print("✅ SAFE")

    print("="*60)


while True:

    text = input("\nEnter text (type 'exit' to quit): ")

    if text.lower() == "exit":
        break

    moderate(text)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


TEXT:
i dont like you
toxic          : 0.7388
severe_toxic   : 0.0081
obscene        : 0.0047
threat         : 0.0026
insult         : 0.0008
identity_hate  : 0.0008

RESULT
🚨 UNSAFE

Detected Categories:
- toxic

BLURRED MESSAGE
████████████████████████

TEXT:
You fucking bastard , pack your crap and get the hell out of here.
toxic          : 0.9989
severe_toxic   : 0.9973
obscene        : 0.9661
threat         : 0.6557
insult         : 0.0186
identity_hate  : 0.0083

RESULT
🚨 UNSAFE

Detected Categories:
- toxic
- severe_toxic
- obscene
- threat

BLURRED MESSAGE
████████████████████████

TEXT:
You Look beautiful today.
toxic          : 0.0004
severe_toxic   : 0.0001
obscene        : 0.0001
threat         : 0.0001
insult         : 0.0001
identity_hate  : 0.0000

RESULT
✅ SAFE


In [ ]:
from transformers import pipeline, AutoModelForSequenceClassification, AutoTokenizer
import torch

MODEL_PATH = "/content/drive/MyDrive/SafeChat_RoBERTa"

LABELS = [
    "toxic",
    "severe_toxic",
    "obscene",
    "threat",
    "insult",
    "identity_hate"
]

# Explicitly load model and tokenizer
model_loaded = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)
tokenizer_loaded = AutoTokenizer.from_pretrained(MODEL_PATH)

classifier = pipeline(
    "text-classification",
    model=model_loaded, # Pass the loaded model object
    tokenizer=tokenizer_loaded, # Pass the loaded tokenizer object
    top_k=None,
    device=0 if torch.cuda.is_available() else -1
)

def moderate(text):

    results = classifier(text)[0]

    print("\n" + "="*60)
    print("TEXT:")
    print(text)
    print("="*60)

    unsafe = False

    detected = []

    for i, item in enumerate(results):

        label = LABELS[i]
        score = item["score"]

        print(f"{label:<15}: {score:.4f}")

        if score > 0.50:
            unsafe = True
            detected.append(label)

    print("\nRESULT")

    if unsafe:
        print("🚨 UNSAFE")

        print("\nDetected Categories:")

        for d in detected:
            print("-", d)

        print("\nBLURRED MESSAGE")
        print("████████████████████████")

    else:
        print("✅ SAFE")

    print("="*60)


while True:

    text = input("\nEnter text (type 'exit' to quit): ")

    if text.lower() == "exit":
        break

    moderate(text)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


Enter text (type 'exit' to quit): smitha

TEXT:
smitha
toxic          : 0.0003
severe_toxic   : 0.0001
obscene        : 0.0001
threat         : 0.0001
insult         : 0.0001
identity_hate  : 0.0000

RESULT
✅ SAFE

Enter text (type 'exit' to quit): i will kill you

TEXT:
i will kill you
toxic          : 0.9770
severe_toxic   : 0.9642
obscene        : 0.1628
threat         : 0.0762
insult         : 0.0588
identity_hate  : 0.0275

RESULT
🚨 UNSAFE

Detected Categories:
- toxic
- severe_toxic

BLURRED MESSAGE
████████████████████████

Enter text (type 'exit' to quit): go to hell

TEXT:
go to hell
toxic          : 0.9956
severe_toxic   : 0.7835
obscene        : 0.5178
threat         : 0.0142
insult         : 0.0051
identity_hate  : 0.0035

RESULT
🚨 UNSAFE

Detected Categories:
- toxic
- severe_toxic
- obscene

BLURRED MESSAGE
████████████████████████

Enter text (type 'exit' to quit): you bitch

TEXT:
you bitch
toxic          : 0.9976
severe_toxic   : 0.9946
obscene        : 0.9838
threat 